# Tutorial: Benchmark From Scratch, ART e Foolbox

**Público:** profissionais que validam implementações adversariais.

**Objetivos:** executar o mesmo modelo/dataset no core e em frameworks opcionais, registrando métricas e indisponibilidades honestamente.

## Roteiro

1. Preparar o modelo comum.
2. Rodar benchmark nativo.
3. Tentar ART/Foolbox sem acoplamento obrigatório.
4. Exercício: comparar paridade.

> Instale requirements-adversarial.txt para habilitar ART/Foolbox.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts" / "python"))
import torch
from torch import nn
from ml_security.framework_benchmark import FrameworkBenchmark

model = nn.Linear(2, 2)
with torch.no_grad():
    model.weight.copy_(torch.tensor([[-10., -10.], [10., 10.]]))
    model.bias.copy_(torch.tensor([10., -10.]))
x = torch.tensor([[.20, .20], [.25, .30], [.80, .80], [.70, .75]])
y = torch.tensor([0, 0, 1, 1])
benchmark = FrameworkBenchmark(ROOT / "results" / "adversarial_benchmark")

## Executar e salvar evidência

A ausência de extras aparece em unavailable; ela não é mascarada nem derruba o benchmark nativo.

In [2]:
report = benchmark.run(model, x, y, epsilon=0.3, include_optional=True)
paths = benchmark.save(report)
{"report": report, "artifacts": paths}

{'report': {'results': [{'framework': 'native',
    'attack': 'fgsm',
    'runtime_ms': 223.142,
    'queries': 0,
    'clean_accuracy': 1.0,
    'robust_accuracy': 0.25,
    'attack_success_rate': 0.75,
    'targeted_success_rate': None,
    'untargeted_success_rate': 0.75,
    'confidence_drop': 0.7262222170829773,
    'l2_distortion': 0.4242640733718872,
    'linf_distortion': 0.30000001192092896}],
  'unavailable': {'art': 'ART não instalado; use requirements-adversarial.txt',
   'foolbox': 'Foolbox não instalado; use requirements-adversarial.txt'}},
 'artifacts': {'json': 'G:\\Outros computadores\\Meu computador\\Controle Base\\Projetos, Robos e Automação\\Projetos Git\\Projetos Extras (Portfolio)\\Ciber\\results\\adversarial_benchmark\\framework_benchmark.json',
  'csv': 'G:\\Outros computadores\\Meu computador\\Controle Base\\Projetos, Robos e Automação\\Projetos Git\\Projetos Extras (Portfolio)\\Ciber\\results\\adversarial_benchmark\\framework_benchmark.csv',
  'png': 'G:\\Outr

## Exercício

Com ART/Foolbox instalados, compare robust accuracy, L2, L∞ e runtime. Investigue diferenças de clipping, loss e defaults antes de concluir que há erro.

**Extensão:** adicione PGD nas três implementações.

In [3]:
[
    {
        key: row.get(key)
        for key in ("framework", "attack", "robust_accuracy", "l2_distortion", "linf_distortion", "runtime_ms")
    }
    for row in report["results"]
]

[{'framework': 'native',
  'attack': 'fgsm',
  'robust_accuracy': 0.25,
  'l2_distortion': 0.4242640733718872,
  'linf_distortion': 0.30000001192092896,
  'runtime_ms': 223.142}]